# Local and Hosted RAG Pipeline using OpenAI

This notebook demonstrates an end-to-end Retrieval-Augmented Generation (RAG) pipeline with OpenAI.

It includes:

- a **custom/local RAG pipeline** using chunking + embeddings + cosine similarity
- a **hosted RAG pipeline** using **OpenAI vector stores** and **file search**
- grounded answer generation with the **Responses API**
- conversational follow-ups with `previous_response_id`

## Topics covered

### Part A — Local custom RAG
- ingest documents
- chunk documents
- generate embeddings
- store vectors locally
- retrieve top-k chunks
- answer questions with grounded prompts

### Part B — Hosted OpenAI RAG
- create a vector store
- upload files
- search a knowledge base
- answer using the `file_search` tool in the Responses API

In [1]:

# %pip install -q openai numpy pandas scikit-learn tiktoken python-dotenv pypdf

import os
import json
import textwrap
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple
from dotenv import load_dotenv, find_dotenv

from openai import OpenAI

_ = load_dotenv(find_dotenv())

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Client initialized.")

Client initialized.


## A small sample corpus

We start with a tiny built-in corpus so the notebook is easy to run immediately.
Later sections show how to load your own `.txt` files and PDFs.

In [3]:

sample_docs = [
    {
        "doc_id": "policy_001",
        "title": "Refund Policy",
        "source": "support_policies/refund_policy.txt",
        "text": '''
        Customers may request a refund within 30 days of purchase for annual subscriptions.
        Monthly subscriptions are generally non-refundable after the billing date, except where required by law.
        Enterprise contracts follow negotiated terms in the master services agreement.
        Refunds are processed back to the original payment method and may take 5 to 10 business days.
        '''
    },
    {
        "doc_id": "policy_002",
        "title": "Data Retention Policy",
        "source": "support_policies/data_retention.txt",
        "text": '''
        User-generated content may be retained for up to 90 days after account deletion for security, abuse prevention,
        and legal compliance purposes. Billing records may be retained longer where required by tax and financial regulations.
        Customers on enterprise plans may negotiate custom retention windows.
        '''
    },
    {
        "doc_id": "policy_003",
        "title": "Security Overview",
        "source": "security/security_overview.txt",
        "text": '''
        All production traffic is encrypted in transit. Sensitive secrets are stored using managed secret storage.
        Administrative access is logged and reviewed. Role-based access control is enforced for internal tools.
        Periodic security assessments are performed, and critical findings are remediated according to severity.
        '''
    },
    {
        "doc_id": "policy_004",
        "title": "Support SLA",
        "source": "support/sla.txt",
        "text": '''
        Enterprise customers receive priority support. Sev-1 incidents are targeted for an initial response within one hour.
        Sev-2 incidents are targeted for an initial response within four hours. Standard tier customers receive support during
        business hours. SLA targets are goals and may vary depending on contract terms.
        '''
    },
]

pd.DataFrame(sample_docs)[["doc_id", "title", "source"]]

,doc_id,title,source
0,policy_001,Refund Policy,support_policies/refund_policy.txt
1,policy_002,Data Retention Policy,support_policies/data_retention.txt
2,policy_003,Security Overview,security/security_overview.txt
3,policy_004,Support SLA,support/sla.txt


# Part A — Local custom RAG

This section shows what happens under the hood in a RAG system.

Pipeline:
1. chunk the documents
2. embed each chunk
3. embed the user question
4. retrieve the most similar chunks
5. generate a grounded answer using only those chunks

In [4]:

@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    title: str
    source: str
    text: str
    start_char: int
    end_char: int

def chunk_text(text: str, chunk_size: int = 250, overlap: int = 50):
    text = " ".join(text.split())
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + chunk_size, n)
        piece = text[start:end].strip()
        if piece:
            chunks.append((piece, start, end))
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

def build_chunks(documents, chunk_size: int = 250, overlap: int = 50):
    all_chunks = []
    for doc in documents:
        for idx, (piece, start, end) in enumerate(chunk_text(doc["text"], chunk_size, overlap)):
            all_chunks.append(
                Chunk(
                    chunk_id=f'{doc["doc_id"]}_chunk_{idx:03d}',
                    doc_id=doc["doc_id"],
                    title=doc["title"],
                    source=doc["source"],
                    text=piece,
                    start_char=start,
                    end_char=end,
                )
            )
    return all_chunks

chunks = build_chunks(sample_docs, chunk_size=180, overlap=40)
pd.DataFrame([asdict(c) for c in chunks]).head(10)

,chunk_id,doc_id,title,source,text,start_char,end_char
0,policy_001_chunk_000,policy_001,Refund Policy,support_policies/refund_policy.txt,Customers may request a refund within 30 days ...,0,180
1,policy_001_chunk_001,policy_001,Refund Policy,support_policies/refund_policy.txt,"the billing date, except where required by law...",140,320
2,policy_001_chunk_002,policy_001,Refund Policy,support_policies/refund_policy.txt,processed back to the original payment method ...,280,361
3,policy_002_chunk_000,policy_002,Data Retention Policy,support_policies/data_retention.txt,User-generated content may be retained for up ...,0,180
4,policy_002_chunk_001,policy_002,Data Retention Policy,support_policies/data_retention.txt,s. Billing records may be retained longer wher...,140,300
5,policy_003_chunk_000,policy_003,Security Overview,security/security_overview.txt,All production traffic is encrypted in transit...,0,180
6,policy_003_chunk_001,policy_003,Security Overview,security/security_overview.txt,nd reviewed. Role-based access control is enfo...,140,315
7,policy_004_chunk_000,policy_004,Support SLA,support/sla.txt,Enterprise customers receive priority support....,0,180
8,policy_004_chunk_001,policy_004,Support SLA,support/sla.txt,geted for an initial response within four hour...,140,315


## Embeddings

This notebook uses `text-embedding-3-small` by default for lower cost.
You can switch to `text-embedding-3-large` for potentially stronger retrieval quality.

In [6]:

EMBEDDING_MODEL = "text-embedding-3-small"

def get_embedding(text: str, model: str = EMBEDDING_MODEL):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

# Quick smoke test:
# vec = get_embedding("hello world")
# len(vec)

In [7]:

def embed_chunks(chunks: List[Chunk], model: str = EMBEDDING_MODEL):
    embeddings = []
    for i, chunk in enumerate(chunks, start=1):
        embeddings.append(get_embedding(chunk.text, model=model))
        if i % 10 == 0 or i == len(chunks):
            print(f"Embedded {i}/{len(chunks)} chunks")
    return np.array(embeddings, dtype=np.float32)

chunk_embeddings = embed_chunks(chunks)
chunk_embeddings.shape

Embedded 9/9 chunks


(9, 1536)

## Retrieve top-k chunks with cosine similarity

In [8]:

def retrieve_top_k(query: str, chunks: List[Chunk], chunk_embeddings: np.ndarray, k: int = 3, model: str = EMBEDDING_MODEL):
    query_vec = np.array(get_embedding(query, model=model), dtype=np.float32).reshape(1, -1)
    sims = cosine_similarity(query_vec, chunk_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:k]

    rows = []
    for rank, idx in enumerate(top_idx, start=1):
        c = chunks[idx]
        rows.append({
            "rank": rank,
            "score": float(sims[idx]),
            "chunk_id": c.chunk_id,
            "doc_id": c.doc_id,
            "title": c.title,
            "source": c.source,
            "text": c.text,
        })
    return pd.DataFrame(rows)

query = "What is the refund window for annual subscriptions and how long do refunds take?"
retrieved_df = retrieve_top_k(query, chunks, chunk_embeddings, k=3)
retrieved_df

,rank,score,chunk_id,doc_id,title,source,text
0,1,0.680214,policy_001_chunk_000,policy_001,Refund Policy,support_policies/refund_policy.txt,Customers may request a refund within 30 days ...
1,2,0.354019,policy_001_chunk_001,policy_001,Refund Policy,support_policies/refund_policy.txt,"the billing date, except where required by law..."
2,3,0.336252,policy_002_chunk_001,policy_002,Data Retention Policy,support_policies/data_retention.txt,s. Billing records may be retained longer wher...


## Generate a grounded answer with the Responses API

The prompt below explicitly asks the model to answer only from retrieved context.

In [10]:

GENERATION_MODEL = "gpt-4.1-mini"

def build_context_block(retrieved_df: pd.DataFrame) -> str:
    parts = []
    for _, row in retrieved_df.iterrows():
        parts.append(
            f'''[Chunk: {row["chunk_id"]}]
Title: {row["title"]}
Source: {row["source"]}
Content: {row["text"]}'''
        )
    return "\n\n".join(parts)

def answer_with_rag(query: str, retrieved_df: pd.DataFrame, model: str = GENERATION_MODEL):
    instructions = '''
    You are a careful retrieval-grounded assistant.
    Use only the supplied retrieved context.
    If the answer is incomplete, say what is missing.
    Cite chunk ids in square brackets at the end of relevant sentences.
    Do not invent policies, numbers, dates, or guarantees.
    '''

    response = client.responses.create(
        model=model,
        instructions=instructions,
        input=f"Question:\n{query}\n\nRetrieved context:\n{build_context_block(retrieved_df)}"
    )
    return response

rag_response = answer_with_rag(query, retrieved_df)
print(rag_response.output_text)

The refund window for annual subscriptions is within 30 days of purchase. Refunds are processed back to the original payment method, but the exact time it takes for refunds to be completed is not specified in the provided information. Additional details on refund processing time are missing from the retrieved context [policy_001_chunk_000, policy_001_chunk_001].


## Inspect retrieved chunks separately

This is important when debugging retrieval quality.

In [11]:

for _, row in retrieved_df.iterrows():
    print("=" * 100)
    print(f'Rank: {row["rank"]} | Score: {row["score"]:.4f} | Chunk: {row["chunk_id"]}')
    print(f'Title: {row["title"]}')
    print(textwrap.fill(row["text"], width=100))

Rank: 1 | Score: 0.6802 | Chunk: policy_001_chunk_000
Title: Refund Policy
Customers may request a refund within 30 days of purchase for annual subscriptions. Monthly
subscriptions are generally non-refundable after the billing date, except where required
Rank: 2 | Score: 0.3540 | Chunk: policy_001_chunk_001
Title: Refund Policy
the billing date, except where required by law. Enterprise contracts follow negotiated terms in the
master services agreement. Refunds are processed back to the original payment m
Rank: 3 | Score: 0.3363 | Chunk: policy_002_chunk_001
Title: Data Retention Policy
s. Billing records may be retained longer where required by tax and financial regulations. Customers
on enterprise plans may negotiate custom retention windows.


## Reusable local RAG class

In [12]:

class LocalRAGPipeline:
    def __init__(
        self,
        client: OpenAI,
        embedding_model: str = "text-embedding-3-small",
        generation_model: str = "gpt-4.1-mini",
        chunk_size: int = 250,
        overlap: int = 50,
    ):
        self.client = client
        self.embedding_model = embedding_model
        self.generation_model = generation_model
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.documents = []
        self.chunks = []
        self.chunk_embeddings = None

    def add_documents(self, documents):
        self.documents.extend(documents)

    def build_index(self):
        self.chunks = build_chunks(self.documents, self.chunk_size, self.overlap)
        self.chunk_embeddings = embed_chunks(self.chunks, self.embedding_model)

    def retrieve(self, query: str, k: int = 3):
        if self.chunk_embeddings is None:
            raise ValueError("Index not built yet. Call build_index() first.")
        return retrieve_top_k(query, self.chunks, self.chunk_embeddings, k=k, model=self.embedding_model)

    def answer(self, query: str, k: int = 3):
        retrieved = self.retrieve(query, k=k)
        response = answer_with_rag(query, retrieved, model=self.generation_model)
        return {"retrieved": retrieved, "response": response}

local_rag = LocalRAGPipeline(client=client)
local_rag.add_documents(sample_docs)
local_rag.build_index()

demo = local_rag.answer("How quickly should Sev-1 and Sev-2 incidents receive an initial response?", k=3)
print(demo["response"].output_text)
display(demo["retrieved"])

Embedded 8/8 chunks
Sev-1 incidents should receive an initial response within one hour. Sev-2 incidents should receive an initial response within four hours. These targets are for enterprise customers and are goals that may vary depending on contract terms [policy_004_chunk_000, policy_004_chunk_001].


,rank,score,chunk_id,doc_id,title,source,text
0,1,0.629513,policy_004_chunk_000,policy_004,Support SLA,support/sla.txt,Enterprise customers receive priority support....
1,2,0.297577,policy_003_chunk_001,policy_003,Security Overview,security/security_overview.txt,nal tools. Periodic security assessments are p...
2,3,0.251523,policy_004_chunk_001,policy_004,Support SLA,support/sla.txt,er customers receive support during business h...


# Part B — Hosted OpenAI RAG with vector stores and file search

This section uses OpenAI-managed retrieval so you do not need to implement chunking and search yourself.

High-level flow:
1. create a vector store
2. upload files
3. let OpenAI ingest them
4. query the knowledge base
5. generate a grounded answer with `file_search`

In [13]:

demo_dir = Path("rag_demo_docs")
demo_dir.mkdir(exist_ok=True)

for doc in sample_docs:
    p = demo_dir / f'{doc["doc_id"]}.txt'
    p.write_text(
        f'Title: {doc["title"]}\nSource: {doc["source"]}\n\n{doc["text"].strip()}',
        encoding="utf-8"
    )

sorted(str(p) for p in demo_dir.glob("*.txt"))

['rag_demo_docs/policy_001.txt',
 'rag_demo_docs/policy_002.txt',
 'rag_demo_docs/policy_003.txt',
 'rag_demo_docs/policy_004.txt']

## Create a vector store and upload files

You can replace these demo files with your own real `.txt`, `.md`, or `.pdf` documents.

In [14]:

vector_store = client.vector_stores.create(name="Notebook RAG Demo Store")
vector_store_id = vector_store.id
print("Vector store created:", vector_store_id)

for file_path in sorted(demo_dir.glob("*.txt")):
    with open(file_path, "rb") as f:
        client.vector_stores.files.upload_and_poll(
            vector_store_id=vector_store_id,
            file=f
        )
    print("Uploaded:", file_path.name)

print("All files uploaded.")

Vector store created: vs_69e29a802278819191160e3ae90f5890
Uploaded: policy_001.txt
Uploaded: policy_002.txt
Uploaded: policy_003.txt
Uploaded: policy_004.txt
All files uploaded.


## Query the vector store directly

This lets you inspect retrieval before generation.

In [16]:

search_results = client.vector_stores.search(
    vector_store_id=vector_store_id,
    query="What are the response targets for critical support incidents?"
)

search_results

SyncPage[VectorStoreSearchResponse](data=[VectorStoreSearchResponse(attributes={}, content=[Content(text='Title: Support SLA\nSource: support/sla.txt\n\nEnterprise customers receive priority support. Sev-1 incidents are targeted for an initial response within one hour.\n        Sev-2 incidents are targeted for an initial response within four hours. Standard tier customers receive support during\n        business hours. SLA targets are goals and may vary depending on contract terms.', type='text')], file_id='file-X1vaJmg4GX3gH5GPntTezR', filename='policy_004.txt', score=0.8821977772260281), VectorStoreSearchResponse(attributes={}, content=[Content(text='Title: Security Overview\nSource: security/security_overview.txt\n\nAll production traffic is encrypted in transit. Sensitive secrets are stored using managed secret storage.\n        Administrative access is logged and reviewed. Role-based access control is enforced for internal tools.\n        Periodic security assessments are performe

## Generate an answer using `file_search` in the Responses API

In [17]:

hosted_response = client.responses.create(
    model="gpt-4.1",
    input="What are the support response targets for Sev-1 and Sev-2 incidents?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_id]
    }]
)

print(hosted_response.output_text)

The support response targets are:

- Sev-1 incidents: Targeted for an initial response within one hour.
- Sev-2 incidents: Targeted for an initial response within four hours.

These targets are for enterprise customers and may vary for other support tiers or depending on contract terms.


## Inspect the full hosted response object

Depending on SDK version, the exact structure may vary. This helps you debug tool results and retrieval traces.

In [18]:

def safe_to_dict(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "to_dict"):
        return obj.to_dict()
    if isinstance(obj, dict):
        return obj
    return json.loads(json.dumps(obj, default=str))

hosted_dict = safe_to_dict(hosted_response)
print(json.dumps(hosted_dict, indent=2)[:12000])

{
  "id": "resp_05adf357829788220069e29c530f90819593d2d170b9d24bc8",
  "created_at": 1776458835.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4.1-2025-04-14",
  "object": "response",
  "output": [
    {
      "id": "fs_05adf357829788220069e29c53b0048195ae44074bd69ca9b0",
      "queries": [
        "support response targets for Sev-1 and Sev-2 incidents",
        "Sev-1 incident response target",
        "Sev-2 incident response target"
      ],
      "status": "completed",
      "type": "file_search_call",
      "results": null
    },
    {
      "id": "msg_05adf357829788220069e29c557b1c81959e9c81fd94307d9d",
      "content": [
        {
          "annotations": [
            {
              "file_id": "file-X1vaJmg4GX3gH5GPntTezR",
              "filename": "policy_004.txt",
              "index": 286,
              "type": "file_citation"
            }
          ],
          "text": "The support response targets are:\n\n- 

## Conversational RAG with `previous_response_id`

This preserves conversational state across turns.

In many real applications, you still rerun retrieval on each turn because a new question may require new evidence.

In [19]:

turn1 = client.responses.create(
    model="gpt-4.1",
    input="Summarize the refund policy in two bullets.",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_id]
    }]
)

print("Turn 1:")
print(turn1.output_text)

turn2 = client.responses.create(
    model="gpt-4.1",
    previous_response_id=turn1.id,
    input="Does that same policy say anything about enterprise contracts?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_id]
    }]
)

print("\nTurn 2:")
print(turn2.output_text)

Turn 1:
- Customers may request a refund within 30 days for annual subscriptions; monthly subscriptions are generally non-refundable after the billing date, except where required by law.
- Refunds are processed back to the original payment method and may take 5 to 10 business days; enterprise contracts follow negotiated terms in the master services agreement.

Turn 2:
Yes, the policy mentions that enterprise contracts follow the negotiated terms outlined in the master services agreement, which may differ from the standard refund policy.


# Common failure modes

- chunks are too big and retrieve irrelevant text
- chunks are too small and split facts apart
- no overlap causes boundary loss
- stale documents get retrieved
- prompts allow the model to guess
- retrieval returns the wrong product, region, or document family
- there is no way to inspect what evidence was used

# Minimal production architecture

A production RAG system often has:

1. **Ingestion pipeline**
   - parse documents
   - clean text
   - chunk
   - embed
   - index
   - attach metadata

2. **Online query path**
   - preprocess query
   - retrieve top-k
   - optionally rerank
   - generate grounded answer
   - return citations

3. **Monitoring**
   - hallucination
   - latency
   - cost per request
   - stale-doc detection

# Cleanup (optional)

Delete the hosted vector store when you no longer need it.

In [20]:

# Uncomment when you are finished:
client.vector_stores.delete(vector_store_id)
print("Deleted vector store:", vector_store_id)

Deleted vector store: vs_69e29a802278819191160e3ae90f5890


# Final notes

This notebook gives you two strong starting points:

- **custom/local RAG** for learning and fine-grained control
- **hosted OpenAI RAG** for fast prototyping with vector stores and file search

Good next extensions:
- metadata filtering
- answer citations rendering
- reranking
- streaming responses
- structured JSON output
- external vector databases